# 03 — DIPI construction

This notebook assembles the final Drought Impact and Pressure Index (DIPI) from the spatially prepared component values sampled at piezometer locations.

The main manuscript variant is:

\[
DIPI_{weighted}=0.4E+0.3P+0.3S
\]

where:
- \(E\) = exposure,
- \(P\) = pressure,
- \(S\) = sensitivity.

For robustness comparison, the notebook also calculates:

\[
DIPI_{equal}=rac{E+P+S}{3}
\]

The component columns are retained as supplied because they represent values sampled from separately prepared GIS rasters. The legacy `dipi` column is preserved only for quality control and is not used in the revised calculations.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT/"data_processed").exists() and (ROOT.parent/"data_processed").exists():
    ROOT = ROOT.parent

PROCESSED = ROOT/"data_processed"
INPUT_FILE = PROCESSED/"DIPI_components.csv"

df = pd.read_csv(INPUT_FILE, float_precision="round_trip")

required = {
    "piezometer_id", "cluster_id", "aquifer_group",
    "x_coord_2180", "y_coord_2180",
    "exposure", "pressure", "sensitivity"
}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

print(f"Input rows: {len(df)}")
print(f"Piezometers: {df['piezometer_id'].nunique()}")

In [ ]:
# Revised DIPI calculations
df["DIPI_weighted"] = (
    0.4 * df["exposure"]
    + 0.3 * df["pressure"]
    + 0.3 * df["sensitivity"]
)

df["DIPI_equal_weight"] = (
    df["exposure"]
    + df["pressure"]
    + df["sensitivity"]
) / 3.0

# Quality-control comparison with the legacy column, when present.
if "dipi" in df.columns:
    df["legacy_dipi_difference"] = df["dipi"] - df["DIPI_weighted"]
    qc = df.loc[
        df["legacy_dipi_difference"].abs() > 1e-6,
        ["piezometer_id", "dipi", "DIPI_weighted", "legacy_dipi_difference"]
    ]
    print(f"Meaningful legacy DIPI discrepancies (>1e-6): {len(qc)}")
    display(qc)
else:
    df["legacy_dipi_difference"] = np.nan


In [ ]:
# Export updated piezometer-level component table.
column_order = [
    "piezometer_id", "cluster_id", "aquifer_group",
    "x_coord_2180", "y_coord_2180",
    "exposure", "vul_norm", "crs_inv_norm", "max_dur_norm",
    "pressure", "qdens_norm", "mining_norm",
    "sensitivity", "quality_norm", "gde_norm",
]

optional_existing = [c for c in column_order if c in df.columns]
output_columns = optional_existing + [
    "DIPI_weighted",
    "DIPI_equal_weight",
    "dipi",
    "legacy_dipi_difference",
]
output_columns = [c for c in output_columns if c in df.columns]

piezometer_output = df[output_columns].copy()
piezometer_path = PROCESSED/"DIPI_components_updated.csv"
piezometer_output.to_csv(piezometer_path, index=False)
print(f"Saved: {piezometer_path}")

In [ ]:
# Aquifer-group summary. This is descriptive only; the manuscript maps are raster-based.
group_summary = (
    df.groupby(["cluster_id", "aquifer_group"], as_index=False)
      .agg(
          n_piezometers=("piezometer_id", "nunique"),
          exposure_mean=("exposure", "mean"),
          pressure_mean=("pressure", "mean"),
          sensitivity_mean=("sensitivity", "mean"),
          DIPI_weighted_mean=("DIPI_weighted", "mean"),
          DIPI_equal_weight_mean=("DIPI_equal_weight", "mean"),
          DIPI_weighted_min=("DIPI_weighted", "min"),
          DIPI_weighted_max=("DIPI_weighted", "max"),
      )
      .sort_values("cluster_id")
      .reset_index(drop=True)
)

group_path = PROCESSED/"DIPI_aquifer_group_summary.csv"
group_summary.to_csv(group_path, index=False)

display(group_summary)
print(f"Saved: {group_path}")

In [ ]:
# Compact quality-control summary
summary = df[[
    "exposure", "pressure", "sensitivity",
    "DIPI_weighted", "DIPI_equal_weight"
]].describe().T

display(summary)

if "legacy_dipi_difference" in df.columns:
    print("Maximum absolute legacy discrepancy:",
          df["legacy_dipi_difference"].abs().max())
